# Lab 4: Model-Based Deep RL

## TDDE78 — Deep Reinforcement Learning
**Linköping University, Spring 2026**

---

This lab covers two model-based RL approaches:

- **Dyna-Q**: augments DQN with a learned neural dynamics model for simulated planning
- **MCTS (UCT)**: uses tree search with environment simulations to select actions

Both methods are evaluated on **CliffWalking-v1** — a 4×12 gridworld where the agent
navigates from start to goal while avoiding a cliff.

> **Note:** The DQN components (Q-network, replay buffer, ε-greedy, Q-update) are already implemented and provided. Your tasks are to implement the **WorldModel** (in `networks.py`) and the **MCTSPlanner**.

In [ ]:
import os
import copy
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from networks import QNetwork, WorldModel
from utils import ReplayBuffer, plot_dyna_results, plot_comparison, smooth, _save_plot

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

_here = globals().get('__vsc_ipynb_file__', os.path.abspath(''))
_nb_dir = os.path.dirname(_here) if os.path.isfile(_here) else _here
EXPERIMENTS_DIR = os.path.normpath(os.path.join(_nb_dir, '..', 'experiments'))
print(f'Experiments directory: {EXPERIMENTS_DIR}')

ENV_NAME  = 'CliffWalking-v1'
N_STATES  = 48   # 4 x 12 grid
N_ACTIONS = 4

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

def obs_to_onehot(obs, n_states=N_STATES):
    """Convert a discrete integer observation to a one-hot float vector."""
    v = np.zeros(n_states, dtype=np.float32)
    v[int(obs)] = 1.0
    return v

set_seed(42)
print('Setup complete!')

---

## A.1 — Explore the Environment

**CliffWalking-v1** is a 4×12 gridworld:
- **Start**: bottom-left (state 36)
- **Goal**: bottom-right (state 47)
- **Cliff**: bottom row between start and goal (states 37–46, reward −100)
- **Reward**: −1 per step, −100 for falling off cliff (reset to start)
- **Solved**: optimal path reward ≈ −13 (go around the top)

The agent has no knowledge of these dynamics — it learns the world model from scratch.

In [ ]:
env = gym.make(ENV_NAME)
print(f'Observation space: {env.observation_space}  (integer state 0–{N_STATES-1})')
print(f'Action space:      {env.action_space}  (0=up, 1=right, 2=down, 3=left)')

obs, _ = env.reset(seed=42)
total, steps = 0, 0
for _ in range(200):
    obs, r, term, trunc, _ = env.step(env.action_space.sample())
    total += r; steps += 1
    if term or trunc: break
print(f'\nRandom agent — steps: {steps}, total reward: {total:.1f}')
print(f'One-hot encoding example: obs={int(obs)} → shape {obs_to_onehot(obs).shape}')
env.close()

---

# Part A — Implementation

---

## DynaAgent (provided skeleton)

> **DQN is reused from Lab 1** — the Q-network, target network, replay buffer, and ε-greedy policy are already implemented in `DynaAgent` below. You do **not** re-implement DQN here.

The new component is the **WorldModel** (in `networks.py`): a neural network trained with supervised learning to predict **next-state logits** (cross-entropy loss), **reward** (MSE loss), and **done** (BCE loss) from (one-hot state, one-hot action).

For discrete environments, the WorldModel outputs **logits over all states** rather than a continuous vector.  During planning the next state is sampled from this learned distribution.

**Key design choice:** Planning only starts after `model_warm_start` real transitions have been collected — this prevents a poorly-fitted model from corrupting the Q-function before it has reliable predictions.

In [ ]:
class DynaAgent:
    """
    Dyna-Q agent: DQN (provided, Lab 1) + WorldModel + simulated planning.

    States are one-hot encoded before storage in the replay buffer.
    The WorldModel predicts next-state logits (cross-entropy) rather than Δs.

    The DQN components are fully provided. Your tasks are:
      1. Implement WorldModel in networks.py
      2. Implement _model_update() below
      3. Implement the planning loop in update()

    Args:
        state_dim        (int):   n_states  (one-hot dimension).
        action_dim       (int):   n_actions.
        lr               (float): Learning rate for Q-network and world model.
        gamma            (float): Discount factor.
        buffer_capacity  (int):   Replay buffer size.
        batch_size       (int):   Mini-batch size.
        planning_steps   (int):   Simulated Q-updates per real step (0 = pure DQN).
        model_warm_start (int):   Real transitions before planning begins.
        target_update    (int):   Hard target-network update frequency (steps).
        max_grad_norm    (float): Gradient clipping for Q-network.
        eps_start        (float): Initial ε for ε-greedy exploration.
        eps_end          (float): Final ε.
        eps_decay        (float): Multiplicative ε decay per episode.
    """

    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99,
                 buffer_capacity=20_000, batch_size=64, planning_steps=10,
                 model_warm_start=500, target_update=100, max_grad_norm=10.0,
                 eps_start=1.0, eps_end=0.05, eps_decay=0.99, seed=42):
        self.state_dim        = state_dim
        self.action_dim       = action_dim
        self.gamma            = gamma
        self.batch_size       = batch_size
        self.planning_steps   = planning_steps
        self.model_warm_start = model_warm_start
        self.target_update    = target_update
        self.max_grad_norm    = max_grad_norm
        self.epsilon          = eps_start
        self.eps_end          = eps_end
        self.eps_decay        = eps_decay
        self._step            = 0

        # ── DQN (reused from Lab 1) ──────────────────────────────────
        self.q_net      = QNetwork(state_dim, action_dim).to(device)
        self.target_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.buffer     = ReplayBuffer(buffer_capacity, state_dim)
        self.q_opt      = torch.optim.Adam(self.q_net.parameters(), lr=lr)

        # ── WorldModel (new — implement in networks.py) ──────────────
        self.world_model = WorldModel(state_dim, action_dim).to(device)
        self.model_opt   = torch.optim.Adam(self.world_model.parameters(), lr=lr)

    # ── Provided: ε-greedy action selection ──────────────────────────
    def select_action(self, state_onehot):
        """ε-greedy action selection on a one-hot encoded state."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.action_dim)
        state_t = torch.FloatTensor(state_onehot).unsqueeze(0).to(device)
        with torch.no_grad():
            return self.q_net(state_t).argmax().item()

    # ── Provided: DQN Q-update ────────────────────────────────────────
    def _q_update(self, states, actions, rewards, next_states, dones):
        """One DQN gradient step with gradient clipping."""
        with torch.no_grad():
            next_q  = self.target_net(next_states).max(dim=1).values
            targets = rewards + self.gamma * (1.0 - dones) * next_q
        q_vals = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        loss   = F.mse_loss(q_vals, targets)
        self.q_opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), self.max_grad_norm)
        self.q_opt.step()
        self._step += 1
        if self._step % self.target_update == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())
        return loss.item()

    # ── TODO: WorldModel supervised update ───────────────────────────
    def _model_update(self, states, actions, rewards, next_states, dones):
        """
        Train the WorldModel on one batch of real transitions.

        Steps:
        1. Convert actions to one-hot:  F.one_hot(actions, self.action_dim).float()
        2. Forward: ns_logits, pred_r, pred_done_logit = self.world_model(states, a_oh)
        3. Losses:
             ns_idx  = next_states.argmax(dim=-1)          # integer index for CE
             s_loss  = F.cross_entropy(ns_logits, ns_idx)  # next-state distribution
             r_loss  = F.mse_loss(pred_r, rewards)
             d_loss  = F.binary_cross_entropy_with_logits(pred_done_logit, dones)
             loss    = s_loss + r_loss + 0.1 * d_loss
        4. Backprop through self.model_opt
        5. Return loss.item()
        """
        raise NotImplementedError('Implement _model_update()')

    # ── TODO: Full Dyna update (real Q-step + model train + planning) ─
    def update(self):
        """
        Full Dyna update:
          1. Return (0.0, 0.0) if buffer has fewer than batch_size transitions.
          2. Sample a batch from the real buffer and move to device.
          3. Q-update on the real batch:     self._q_update(...)
          4. Model update on the real batch: self._model_update(...)
          5. Planning loop — only if len(self.buffer) >= self.model_warm_start:
               repeat planning_steps times:
                 a. Sample states/actions from the real buffer
                 b. With torch.no_grad():
                      a_oh = F.one_hot(a_b, self.action_dim).float()
                      ns_logits, r_sim, done_logit = self.world_model(s_b, a_oh)
                      ns_idx_sim = Categorical(logits=ns_logits).sample()
                      ns_sim     = F.one_hot(ns_idx_sim, self.state_dim).float()
                      done_sim   = (sigmoid(done_logit) > 0.5).float()
                 c. Q-update on synthetic transitions: self._q_update(...)
          6. Return (q_loss, model_loss)
        """
        raise NotImplementedError('Implement update()')

print('DynaAgent defined')

In [ ]:
def train_dyna(env_name=ENV_NAME, total_timesteps=50_000,
               planning_steps=10, seed=42, solve_threshold=-14.0,
               log_interval=50, **agent_kwargs):
    """
    Train a DynaAgent on CliffWalking-v1.

    With planning_steps=0 this is equivalent to a standard DQN (baseline).

    Args:
        env_name        (str):   Gymnasium environment id.
        total_timesteps (int):   Maximum environment steps.
        planning_steps  (int):   Simulated updates per real step.
        seed            (int):   Random seed.
        solve_threshold (float): Early-stop when avg reward (last 50 ep) >= this.
        log_interval    (int):   Print every N episodes.
        **agent_kwargs:          Passed to DynaAgent.

    Returns:
        dict with keys: episode_rewards, episode_timesteps, q_losses, model_losses, agent
    """
    set_seed(seed)
    # max_episode_steps=200 guarantees truncation; some Gymnasium builds of
    # CliffWalking-v1 omit the TimeLimit wrapper, leaving episodes open-ended.
    env       = gym.make(env_name, max_episode_steps=200)
    n_states  = env.observation_space.n
    n_actions = env.action_space.n

    agent = DynaAgent(n_states, n_actions,
                      planning_steps=planning_steps, seed=seed, **agent_kwargs)

    episode_rewards   = []
    episode_timesteps = []
    q_losses          = []
    model_losses      = []

    raw_obs, _ = env.reset(seed=seed)
    obs        = obs_to_onehot(raw_obs, n_states)
    ep_reward  = 0.0

    for step in range(total_timesteps):
        action                           = agent.select_action(obs)
        next_raw, reward, term, trunc, _ = env.step(action)
        done                             = term or trunc
        next_obs                         = obs_to_onehot(next_raw, n_states)

        agent.buffer.push(obs, action, reward, next_obs, float(term))
        ep_reward += reward
        obs = next_obs

        if done:
            episode_rewards.append(ep_reward)
            episode_timesteps.append(step + 1)
            ep_reward  = 0.0
            raw_obs, _ = env.reset()
            obs        = obs_to_onehot(raw_obs, n_states)
            agent.epsilon = max(agent.eps_end, agent.epsilon * agent.eps_decay)

            n = len(episode_rewards)
            if n % log_interval == 0:
                avg = np.mean(episode_rewards[-log_interval:])
                print(f'Step {step+1:>6,} | Episode {n:4d} | '
                      f'Avg({log_interval}): {avg:6.1f} | eps: {agent.epsilon:.3f}')

            if solve_threshold and n >= 50:
                if np.mean(episode_rewards[-50:]) >= solve_threshold:
                    print(f'Solved at step {step+1:,}!')
                    break

        q_loss, m_loss = agent.update()
        q_losses.append(q_loss)
        model_losses.append(m_loss)

    env.close()
    return {
        'episode_rewards':   episode_rewards,
        'episode_timesteps': episode_timesteps,
        'q_losses':          q_losses,
        'model_losses':      model_losses,
        'agent':             agent,
    }

print('train_dyna defined')

## A.2 — Train Dyna on CliffWalking-v1

Train DynaAgent with `planning_steps=10`. The solve threshold is −14
(optimal path reward ≈ −13). Compare against the DQN baseline (`planning_steps=0`).

In [ ]:
set_seed(42)

results_dyna = train_dyna(
    env_name        = ENV_NAME,
    total_timesteps = 50_000,
    planning_steps  = 10,
    seed            = 42,
    lr              = 1e-3,
    batch_size      = 64,
    solve_threshold = -14.0,
    log_interval    = 50,
)

plot_dyna_results(results_dyna, title='Dyna — CliffWalking-v1', experiments_dir=EXPERIMENTS_DIR)
print(f'Final avg reward (last 20 ep): {np.mean(results_dyna["episode_rewards"][-20:]):.1f}')

## MCTS Planner

**Monte Carlo Tree Search (UCT)** plans by building a search tree from the current state. Each simulation:

1. **Selection** — traverse from root by UCB1 score until a leaf node
2. **Expansion** — add all unvisited children
3. **Simulation** — random rollout from the expanded node
4. **Backpropagation** — update visit counts and value estimates up to root

The action with the highest visit count at the root is returned.

We use `copy.deepcopy(env)` to fork the environment for each simulation — the real environment is never modified.

In [ ]:
class MCTSNode:
    """A single node in the MCTS tree — provided, no changes needed."""
    __slots__ = ('parent', 'action', 'children', 'visit_count', 'value_sum')

    def __init__(self, parent=None, action=None):
        self.parent      = parent
        self.action      = action
        self.children    = {}   # action_int -> MCTSNode
        self.visit_count = 0
        self.value_sum   = 0.0

    @property
    def q_value(self):
        return self.value_sum / self.visit_count if self.visit_count > 0 else 0.0

    def ucb(self, c=1.41):
        if self.visit_count == 0:
            return float('inf')
        return self.q_value + c * np.sqrt(np.log(self.parent.visit_count) / self.visit_count)

    def is_leaf(self):
        return len(self.children) == 0


class MCTSPlanner:
    """
    UCT planner using the true Gymnasium environment as a model.

    Uses copy.deepcopy(env) to fork the environment for each simulation.

    Args:
        action_dim    (int):   Number of discrete actions.
        n_simulations (int):   MCTS iterations per action selection.
        depth         (int):   Maximum rollout depth per simulation.
        gamma         (float): Discount factor for rollout returns.
        c             (float): UCB exploration constant.
    """

    def __init__(self, action_dim, n_simulations=50, depth=30, gamma=1.0, c=1.41):
        self.action_dim    = action_dim
        self.n_simulations = n_simulations
        self.depth         = depth
        self.gamma         = gamma
        self.c             = c

    def select_action(self, env):
        """
        Run MCTS from the current env state and return the best action.

        Args:
            env: current Gymnasium env (deepcopied internally — not modified).

        Returns:
            int: action with highest visit count at root.

        Steps (repeat n_simulations times):
        1. sim_env = copy.deepcopy(env); node = root; G = 0.0; disc = 1.0; depth = 0
        2. Selection: while not node.is_leaf() and not done and depth < self.depth:
               pick child with max UCB, step sim_env, accumulate G, depth += 1
        3. Expansion: if not done and depth < self.depth:
               add all action children to node;
               pick a random child, step sim_env, accumulate G, depth += 1
        4. Simulation (random rollout): step sim_env with random actions
               until done or depth >= self.depth, accumulate G
        5. Backpropagation: walk from node to root:
               n.visit_count += 1; n.value_sum += G
        Return: max(root.children, key=lambda a: root.children[a].visit_count)
        """
        # =====================================================================
        # TODO: Implement UCT search.
        #
        # Hints:
        #   root = MCTSNode()
        #   root.visit_count = 1   # avoids log(0) in UCB of direct children
        #   for _ in range(self.n_simulations):
        #       sim_env = copy.deepcopy(env)
        #       ... (see docstring above)
        #   if not root.children: return np.random.randint(self.action_dim)
        # =====================================================================
        raise NotImplementedError('Implement MCTSPlanner.select_action()')

print('MCTSNode and MCTSPlanner defined')

In [ ]:
def evaluate_mcts(env_name=ENV_NAME, n_simulations=50, depth=30,
                  num_episodes=20, seed=0):
    """
    Evaluate an MCTSPlanner on a given environment.

    Args:
        env_name      (str): Gymnasium environment id.
        n_simulations (int): MCTS iterations per step.
        depth         (int): Rollout depth per simulation.
        num_episodes  (int): Number of evaluation episodes.
        seed          (int): Base random seed.

    Returns:
        list: Episode rewards.
    """
    env     = gym.make(env_name)
    planner = MCTSPlanner(env.action_space.n,
                          n_simulations=n_simulations, depth=depth)
    rewards = []

    for ep in range(num_episodes):
        obs, _ = env.reset(seed=seed + ep)
        ep_reward = 0.0
        for _ in range(500):
            action = planner.select_action(env)
            obs, r, term, trunc, _ = env.step(action)
            ep_reward += r
            if term or trunc:
                break
        rewards.append(ep_reward)
        print(f'Episode {ep+1:2d}/{num_episodes}: reward = {ep_reward:.1f}')

    env.close()
    print(f'\nMean +/- Std: {np.mean(rewards):.1f} +/- {np.std(rewards):.1f}')
    return rewards

print('evaluate_mcts defined')

## A.3 — Evaluate MCTS on CliffWalking-v1

Run MCTS with `n_simulations=50` and `depth=30` for 20 episodes.
Note: MCTS does **not** learn — it plans from scratch at each step using the true env.

In [ ]:
## B.1 — Ablation: Planning Steps in Dyna

Sweep `planning_steps` ∈ {0, 5, 20, 50}. `k=0` is the pure DQN baseline.
At what point does more planning stop helping (model error compounding)?

seeds    = [42, 123, 456]
k_values = [0, 5, 20, 50]

ablation_results = {}
for k in k_values:
    ablation_results[f'k = {k}'] = [
        train_dyna(env_name=ENV_NAME, total_timesteps=50_000,
                   planning_steps=k, seed=s, lr=1e-3, batch_size=64,
                   solve_threshold=None)
        for s in seeds
    ]

# Use global min_eps so every k-value is plotted over the same episode range.
# (k=0 has fewest episodes; k=50 may have many more due to faster convergence.)
global_min_eps = min(
    min(len(r['episode_rewards']) for r in runs)
    for runs in ablation_results.values()
)

window  = 20
colors  = plt.rcParams['axes.prop_cycle'].by_key()['color']
fig, ax = plt.subplots(figsize=(12, 5))

for idx, (name, runs) in enumerate(ablation_results.items()):
    mat      = np.array([r['episode_rewards'][:global_min_eps] for r in runs])
    mean_r   = mat.mean(0)
    std_r    = mat.std(0)
    sm_mean  = smooth(mean_r, window)
    sm_std   = smooth(std_r,  window)
    episodes = np.arange(window, global_min_eps + 1)
    color    = colors[idx % len(colors)]
    ax.plot(episodes, sm_mean, color=color, label=name, linewidth=2)
    ax.fill_between(episodes, sm_mean - sm_std, sm_mean + sm_std,
                    color=color, alpha=0.15)

ax.set_xlabel('Episode')
ax.set_ylabel('Episode Reward')
ax.set_ylim(None, 0)
ax.set_title('Dyna — Planning Steps Ablation (CliffWalking-v1)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
_save_plot(fig, 'Dyna Planning Steps Ablation CliffWalking', EXPERIMENTS_DIR)
plt.show()

print(f'All lines shown over first {global_min_eps} episodes (limited by k=0 baseline).')
print('Mean +/- Std (last 20 episodes):')
for name, runs in ablation_results.items():
    final = [np.mean(r['episode_rewards'][-20:]) for r in runs]
    print(f'  {name}: {np.mean(final):.1f} +/- {np.std(final):.1f}')

## B.2 — MCTS: Effect of Simulation Count

Sweep `n_simulations` ∈ {5, 20, 50, 200}.
What is the minimum number of simulations needed to reliably find the safe path?

In [ ]:
sim_counts = [5, 20, 50, 200]
seeds_mcts = [0, 1, 2]

mcts_results = {}
for n in sim_counts:
    episode_lists = []
    for s in seeds_mcts:
        ep_rewards = evaluate_mcts(
            env_name      = ENV_NAME,
            n_simulations = n,
            depth         = 30,
            num_episodes  = 10,
            seed          = s * 100,
        )
        episode_lists.append(ep_rewards)
    mcts_results[f'n_sim = {n}'] = episode_lists

labels = list(mcts_results.keys())
means  = [np.mean([np.mean(ep) for ep in runs]) for runs in mcts_results.values()]
stds   = [np.std( [np.mean(ep) for ep in runs]) for runs in mcts_results.values()]

# Anchor bars below the worst value so they grow upward:
# taller bar = closer to 0 = better performance.
y_bot   = min(m - s for m, s in zip(means, stds)) - 5
heights = [m - y_bot for m in means]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(labels, heights, bottom=y_bot, yerr=stds, capsize=5,
       color='steelblue', alpha=0.7)
ax.axhline(-13, color='green', linestyle=':', label='Optimal = -13')
ax.set_ylim(y_bot, 0)
ax.set_ylabel('Mean Episode Reward')
ax.set_xlabel('Simulations per Step')
ax.set_title('MCTS — Effect of Simulation Count on CliffWalking-v1')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
_save_plot(fig, 'MCTS Simulation Count CliffWalking', EXPERIMENTS_DIR)
plt.show()

print('Mean +/- Std episode reward by simulation count:')
for k, (m, s) in zip(labels, zip(means, stds)):
    print(f'  {k}: {m:.1f} +/- {s:.1f}')

## B.2 — MCTS: Effect of Simulation Count

Sweep `n_simulations` ∈ {5, 20, 50, 200}.
More simulations = better planning but higher compute cost per step.
What is the minimum number of simulations needed to solve CartPole reliably?

In [ ]:
sim_counts = [5, 20, 50, 200]
seeds_mcts = [0, 1, 2]

mcts_results = {}
for n in sim_counts:
    episode_lists = []
    for s in seeds_mcts:
        ep_rewards = evaluate_mcts(
            env_name='CartPole-v1',
            n_simulations=n, depth=20,
            num_episodes=10, seed=s * 100,
        )
        episode_lists.append(ep_rewards)
    mcts_results[f'n_sim = {n}'] = episode_lists

# Summary bar chart
labels = list(mcts_results.keys())
means  = [np.mean([np.mean(ep) for ep in runs]) for runs in mcts_results.values()]
stds   = [np.std( [np.mean(ep) for ep in runs]) for runs in mcts_results.values()]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, means, yerr=stds, capsize=5, color='steelblue', alpha=0.7)
ax.set_ylabel('Mean Episode Reward'); ax.set_xlabel('Simulations per Step')
ax.set_title('MCTS — Effect of Simulation Count on CartPole-v1')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
from utils import _save_plot
_save_plot(fig, 'MCTS Simulation Count', EXPERIMENTS_DIR)
plt.show()

print('Mean ± Std episode reward by simulation count:')
for k, (m, s) in zip(labels, zip(means, stds)):
    print(f'  {k}: {m:.1f} ± {s:.1f}')